# People Like Me

Setup:  create a .venv -- must be in a workspace to do this.  This .venv will be available to all projects in workspace
Use:  https://chatgpt.com/share/69cd1090-2a98-8328-9f28-b24fa7b362ce

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# print("Path to dataset files:", path)

In [ ]:
# From Hugging Face:pip install datasets
# Dataset is located here:  C:\Users\DeloresMincarelli\.cache\huggingface\hub\

import pandas as pd
from datasets import load_dataset

dataset = load_dataset("med2425/resume-job-fit-merged-v1")

In [ ]:
import json

row = dataset["train"][0]
print(json.dumps(row, indent=2, ensure_ascii=False))

In [ ]:

train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()


train.head()
# row = train.iloc[0].to_dict()
# print(json.dumps(row, indent=2, ensure_ascii=False))

In [ ]:
distinct_resume_domains = sorted(train["resume_domain"].dropna().unique())
distinct_resume_domains

In [ ]:


train_resume = train[["resume", "resume_domain"]].copy()
test_resume = test[["resume", "resume_domain"]].copy()

train_resume = train_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)
test_resume = test_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)

# One ID per distinct resume text (shared across train/test)
combined_resume = pd.concat([train_resume["resume"], test_resume["resume"]], ignore_index=True)
resume_ids, _ = pd.factorize(combined_resume)
n_train = len(train_resume)
train_resume["resume_id"] = resume_ids[:n_train]
test_resume["resume_id"] = resume_ids[n_train:]

train_resume["source"] = "huggingface"
test_resume["source"] = "huggingface"

train_resume = train_resume[["resume_id", "resume", "resume_domain", "source"]]
test_resume = test_resume[["resume_id", "resume", "resume_domain", "source"]]

train_resume.head()

### Add your resume from a PDF

Install once: `pip install pypdf ipywidgets`

1. Run the next cell (extract helpers).
2. Run the cell after that (optional path + upload widget). If you use **Upload**, pick your PDF, then run the **following** cell again so the widget value is read.
3. Run the last cell to extract text, save it beside the notebook as `my_resume_extracted.txt`, and append one row to `train` (`resume_domain='data'`). Then **re-run** the `train_resume` build cell so your resume is included there too.

In [ ]:
import io
import re
from pathlib import Path

from pypdf import PdfReader


def organize_resume_text(text: str) -> str:
    """Normalize whitespace and page breaks for cleaner plain text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(data: bytes) -> str:
    reader = PdfReader(io.BytesIO(data))
    parts = []
    for page in reader.pages:
        t = page.extract_text()
        if t:
            parts.append(t.strip())
    return organize_resume_text("\n\n".join(parts))


print("Ready: organize_resume_text, extract_text_from_pdf")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
# PDF in the same folder as the notebook (when Jupyter cwd is this folder)
MY_RESUME_PDF = NOTEBOOK_DIR / "dam_resume.pdf"

print(f"If no upload button appears (common in Cursor), use a PDF at:\n  {MY_RESUME_PDF.resolve()}\n")

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Resume PDF")
display(pdf_upload)

In [ ]:
import pandas as pd


def _pdf_bytes_from_upload(w) -> bytes | None:
    if not w.value:
        return None
    f0 = w.value[0]
    content = f0["content"]
    if hasattr(content, "tobytes"):
        return content.tobytes()
    return bytes(memoryview(content))


pdf_bytes = _pdf_bytes_from_upload(pdf_upload)
if pdf_bytes is not None:
    print("Using PDF from upload widget.")
elif MY_RESUME_PDF.exists():
    pdf_bytes = MY_RESUME_PDF.read_bytes()
    print(f"Using PDF from disk: {MY_RESUME_PDF.resolve()}")
else:
    raise FileNotFoundError(
        f"No PDF yet: use the upload widget and re-run this cell, "
        f"or save your file as:\n  {MY_RESUME_PDF.resolve()}"
    )

my_resume_text = extract_text_from_pdf(pdf_bytes)
RESUME_TEXT_CACHE = NOTEBOOK_DIR / "my_resume_extracted.txt"
RESUME_TEXT_CACHE.write_text(my_resume_text, encoding="utf-8")

new_row = pd.DataFrame(
    [
        {
            "resume": my_resume_text,
            "jd": "",
            "label": pd.NA,
            "source": "huggingface",
            "resume_domain": "data",
            "jd_domain": pd.NA,
        }
    ]
)
train = pd.concat([train, new_row], ignore_index=True)

print(f"Saved text to {RESUME_TEXT_CACHE} ({len(my_resume_text):,} chars). train rows: {len(train)}")